# 01 — `midas-propagate`: calibration-aware per-grain covariance → stress

Production HEDM tools report per-grain σ with the detector calibration
**held fixed**. `midas-propagate` closes that loop: it propagates
calibration uncertainty into per-grain covariance and then into per-grain
**stress** error bars.

> **Status note.** The package README labels this a scaffold, but the
> three core numerical modules below are implemented and unit-tested
> (`tests/`). This notebook exercises them end to end on synthetic data.
> The full pipeline glue (reading a real MIDAS dataset) is still in
> progress.

The three stages, mirroring `tests/`:

1. **`joint_nll`** — per-grain Hessian blocks `H_gg`, `H_gc` of the joint
   negative-log-likelihood on grain state `g` and calibration `c`.
2. **`schur`** — Schur-complement marginalisation: turn `(H_gg, H_gc,
   Σ_cc)` into a calibration-*marginalised* per-grain covariance
   `Σ_gg`, which is wider than the calibration-frozen one.
3. **`propagate`** — delta-method `Σ_σ = J Σ_g Jᵀ` from per-grain
   covariance to per-grain Cauchy-stress covariance via a known
   single-crystal stiffness.

CPU + synthetic; runs in a few seconds.


In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import math
import numpy as np
import torch

torch.manual_seed(0)
torch.set_default_dtype(torch.float64)
print("torch", torch.__version__, "| device: cpu")
import midas_propagate as mpr
print("midas_propagate", mpr.__version__)


torch 2.11.0 | device: cpu
midas_propagate 0.0.0.dev0


## Stage 1 — Per-grain Hessian blocks from the joint NLL

Forward-simulate one FCC Au grain on a paper-1 FF geometry, treat its
predicted spots as the observation (so the residual is zero at the MAP),
and compute the two Hessian blocks:

- `H_gg` — grain-state curvature `(12, 12)` on `(euler[3], latc[6], pos[3])`.
- `H_gc` — grain↔calibration coupling `(12, n_c)` for the chosen
  calibration parameters.

`apply_tilts=True` is required so the forward model uses `ty`/`tz`
directly, letting calibration tilt uncertainty couple into the grain
residual.


In [2]:
from midas_diffract import HEDMForwardModel, HEDMGeometry, hkls_for_forward_model
from midas_hkls import SpaceGroup, Lattice
from midas_propagate.joint_nll import GrainObs, per_grain_hessian_blocks

DEG2RAD = math.pi / 180.0
geom = HEDMGeometry(
    Lsd=1_000_000.0, y_BC=1024.0, z_BC=1024.0, px=200.0,
    omega_start=0.0, omega_step=0.25, n_frames=1440,
    n_pixels_y=2048, n_pixels_z=2048,
    min_eta=6.0, wavelength=0.172979, apply_tilts=True,
)
sg = SpaceGroup.from_number(225)
lat = Lattice.for_system("cubic", a=4.08)
hkls_cart, thetas, hkls_int = hkls_for_forward_model(
    sg, lat, wavelength_A=geom.wavelength, two_theta_max_deg=15.0,
)
model = HEDMForwardModel(hkls=hkls_cart, thetas=thetas,
                         geometry=geom, hkls_int=hkls_int)

gt_euler = torch.tensor([45.0, 30.0, 60.0]) * DEG2RAD
gt_latc  = torch.tensor([4.08, 4.08, 4.08, 90.0, 90.0, 90.0])
gt_pos   = torch.zeros(3)

spots = model(gt_euler.unsqueeze(0), gt_pos.unsqueeze(0), lattice_params=gt_latc)
det, valid = HEDMForwardModel.predict_spot_coords(spots, space="detector")
obs = det.squeeze()[valid.squeeze() > 0.5].detach().clone()
print("observed spots:", obs.shape[0])


observed spots: 158


In [3]:
grain_obs = GrainObs(
    spot_id=0, euler_rad=gt_euler, latc=gt_latc, pos_um=gt_pos,
    observed_detector=obs,
)
calibration_names = ["Lsd", "BC_y", "BC_z", "ty", "tz"]
calibration_map = torch.tensor([1_000_000.0, 1024.0, 1024.0, 0.0, 0.0])
sigma_obs_detector = torch.full((3,), 0.5)         # px / frame measurement noise

blocks = per_grain_hessian_blocks(
    grain_obs,
    hkls_cart=hkls_cart, hkls_int=hkls_int, thetas=thetas,
    base_geometry=geom, scan_config=None,
    calibration_names=calibration_names,
    calibration_map=calibration_map,
    sigma_obs_detector=sigma_obs_detector,
    method="fisher",
)
print("H_gg:", tuple(blocks.H_gg.shape), "| H_gc:", tuple(blocks.H_gc.shape),
      "| spots matched:", blocks.n_spots_matched)
eig = torch.linalg.eigvalsh(0.5 * (blocks.H_gg + blocks.H_gg.T))
print("H_gg smallest eigenvalue (PSD check):", float(eig.min()))
col_norms = torch.linalg.norm(blocks.H_gc, dim=0)
print("H_gc column norms:", {n: round(float(v), 3) for n, v in zip(calibration_names, col_norms)})
print("\nLsd/ty/tz carry the most grain<->calibration coupling, as expected.")


H_gg: (12, 12) | H_gc: (12, 5) | spots matched: 158
H_gg smallest eigenvalue (PSD check): 0.007703344440202754
H_gc column norms: {'Lsd': 76.794, 'BC_y': 2.127, 'BC_z': 24608.441, 'ty': 12319.908, 'tz': 3.56}

Lsd/ty/tz carry the most grain<->calibration coupling, as expected.


## Stage 2 — Schur-marginalised per-grain covariance

`per_grain_schur_marginal(H_gg, H_gc, Σ_cc)` returns two per-grain
covariances:

- `sigma_gg_frozen = H_gg⁻¹` — calibration held fixed (what current tools
  report).
- `sigma_gg_calmarg` — calibration uncertainty marginalised in. It is
  **wider** (the inflation eigenvalues are ≥ 1), and that inflation is
  exactly the calibration's contribution to per-grain uncertainty.

The PSD-inflation guarantee holds when each grain's implied joint
Hessian `[[H_gg, H_gc], [H_gcᵀ, Σ_cc⁻¹]]` is PSD — i.e. the
grain↔calibration coupling is bounded by `H_gg`'s spectral floor (the
real-HEDM regime; see `tests/test_schur.py`). To demonstrate the property
cleanly we use well-conditioned synthetic blocks at paper-1 scale; the
*shapes* match the real `(12, n_c)` blocks computed in Stage 1.


In [4]:
from midas_propagate.schur import per_grain_schur_marginal, sigma_inflation_ratio

torch.manual_seed(1)
G, n_g, n_c = 50, 12, len(calibration_names)

# Calibration posterior covariance Σ_cc (SPD) — stand-in for the
# midas-calibrate-v2 posterior over (Lsd, BC_y, BC_z, ty, tz).
Bc = torch.randn(n_c, n_c)
sigma_cc = Bc @ Bc.T + 0.5 * torch.eye(n_c)

# Per-grain SPD H_gg with a strong diagonal floor, and a small H_gc so the
# joint Hessian stays PSD (the bounded-coupling regime).
A = torch.randn(G, n_g, n_g)
H_gg = A @ A.transpose(-1, -2) + 5.0 * torch.eye(n_g)
H_gc = torch.randn(G, n_g, n_c) * 0.05

res = per_grain_schur_marginal(H_gg, H_gc, sigma_cc, ridge_g=1e-12)
ratio = sigma_inflation_ratio(res.sigma_gg_frozen, res.sigma_gg_calmarg)
print("frozen  σ (grain 0, first 3 diag):",
      np.round(torch.sqrt(torch.diagonal(res.sigma_gg_frozen[0])[:3]).tolist(), 5))
print("calmarg σ (grain 0, first 3 diag):",
      np.round(torch.sqrt(torch.diagonal(res.sigma_gg_calmarg[0])[:3]).tolist(), 5))
print("inflation eigenvalues all >= 1:", bool((ratio >= 1.0 - 1e-9).all()),
      "| max inflation:", round(float(ratio.max()), 4))
print("\n=> calibration uncertainty provably widens every grain's covariance.")


frozen  σ (grain 0, first 3 diag): [0.36773 0.28828 0.30642]
calmarg σ (grain 0, first 3 diag): [0.36984 0.28863 0.3072 ]
inflation eigenvalues all >= 1: True | max inflation: 1.0246

=> calibration uncertainty provably widens every grain's covariance.


### Sanity: zero coupling ⇒ no inflation

If a grain doesn't couple to calibration (`H_gc = 0`), the marginalised
covariance equals the frozen one exactly — calibration uncertainty
cannot leak in.


In [5]:
res0 = per_grain_schur_marginal(
    H_gg=blocks.H_gg.unsqueeze(0),
    H_gc=torch.zeros_like(blocks.H_gc).unsqueeze(0),
    sigma_cc=sigma_cc, ridge_g=1e-10,
)
gap = (res0.sigma_gg_calmarg - res0.sigma_gg_frozen).abs().max()
print("max |calmarg - frozen| with H_gc=0:", float(gap), "(≈ 0 as expected)")


max |calmarg - frozen| with H_gc=0: 0.0 (≈ 0 as expected)


## Stage 3 — Delta-method to per-grain stress covariance

Finally, propagate per-grain covariance through Hooke's law to per-grain
Cauchy stress and its 6×6 Voigt covariance: `Σ_σ = J Σ_g Jᵀ`, with
`J = ∂σ(g)/∂g`. Position drops out (stress depends only on orientation +
lattice). We use a cubic single-crystal stiffness.


In [6]:
from midas_propagate.propagate import per_grain_stress_with_cov

def cubic_stiffness(C11=200.0, C12=130.0, C44=80.0):
    C = torch.tensor([
        [C11, C12, C12, 0, 0, 0],
        [C12, C11, C12, 0, 0, 0],
        [C12, C12, C11, 0, 0, 0],
        [0, 0, 0, 2 * C44, 0, 0],     # Mandel: 2*C44 on shear diagonal
        [0, 0, 0, 0, 2 * C44, 0],
        [0, 0, 0, 0, 0, 2 * C44],
    ], dtype=torch.float64)
    return C

# One grain: slightly strained orientation; use grain 0's
# calibration-marginalised covariance from Stage 2 as Σ_g.
g_map = torch.cat([gt_euler, gt_latc, gt_pos]).unsqueeze(0)        # (1, 12)
# Bump the lattice slightly so there is a non-zero stress to report.
g_map[0, 3:6] = torch.tensor([4.083, 4.080, 4.079])
sigma_gg = res.sigma_gg_calmarg[:1]                                # (1, 12, 12)
latc_ref = torch.tensor([4.08, 4.08, 4.08, 90.0, 90.0, 90.0])
C = cubic_stiffness()

stress = per_grain_stress_with_cov(g_map, sigma_gg, latc_ref, C)
print("per-grain stress (Voigt, GPa):", np.round(stress.stress_voigt.squeeze(0).tolist(), 4))
print("per-grain stress σ (Voigt, GPa):", np.round(stress.sigma_voigt.squeeze(0).tolist(), 5))
# PSD check on the stress covariance.
cov = stress.stress_cov.squeeze(0)
print("stress-cov min eigenvalue (PSD):",
      float(torch.linalg.eigvalsh(0.5 * (cov + cov.T)).min()))


per-grain stress (Voigt, GPa): [ 0.0632  0.1018  0.0605 -0.0083 -0.013   0.0353]
per-grain stress σ (Voigt, GPa): [17.92501 19.48703 19.83429  2.8693   3.16447  3.67511]
stress-cov min eigenvalue (PSD): 0.1665966854958737


## Summary

End to end on synthetic data:

1. `joint_nll.per_grain_hessian_blocks` → `H_gg`, `H_gc` (PSD `H_gg`,
   Lsd/ty/tz dominate the coupling).
2. `schur.per_grain_schur_marginal` → calibration-marginalised per-grain
   covariance, provably wider than the frozen one; zero coupling gives no
   inflation.
3. `propagate.per_grain_stress_with_cov` → per-grain stress with a PSD
   Voigt covariance via the delta method.

This is the paper-1 chain "calibration σ → grain σ → stress σ". The
remaining work (tracked in `dev/paper/SKETCH.md`) is wiring it to real
MIDAS Grains.csv / paramstest inputs and a measured `Σ_cc` from
`midas-calibrate-v2`.
